# Lesson 6.3: Errors and Power

Simulate thousands of experiments to see the Type I error rate when nothing is going on and the power when a real 6-point effect exists, then check with a power calculator.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Plan the experiment

A teaching method may raise scores by 6 points (SD 15). We compare two groups of 36 students with a t-test at alpha = 0.05. The effect size d is the gap measured in SDs: 6 / 15 = 0.4.

In [1]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
n, sd, diff, alpha, reps = 36, 15, 6, 0.05, 10_000

d = diff / sd
print(f'effect size d = {d:.2f}')

effect size d = 0.40


## 2. Simulate when the null is true

We run 10,000 pretend experiments where both groups come from the same population. The share of experiments that still reject H0 is the Type I error rate, and it should be close to alpha.

In [2]:
def reject_rate(true_diff):
    a = rng.normal(0, sd, size=(reps, n))
    b = rng.normal(true_diff, sd, size=(reps, n))
    p = stats.ttest_ind(a, b, axis=1).pvalue   # one t-test per row
    return (p <= alpha).mean()

type1 = reject_rate(0)
print(f'Type I error rate (no real effect) = {type1:.3f}')

Type I error rate (no real effect) = 0.049


## 3. Simulate when a real effect exists

Now the second group truly scores 6 points higher. The share of experiments that reject H0 is the power. With only 36 per group the test misses the effect more often than it finds it.

In [3]:
power_sim = reject_rate(diff)

print(f'simulated power = {power_sim:.3f}')
print(f'simulated Type II error (beta) = {1 - power_sim:.3f}')

simulated power = 0.387
simulated Type II error (beta) = 0.613


## 4. Check with statsmodels and plan the sample size

TTestIndPower does the calculation exactly, without simulation. power() gives the power for n = 36 per group, and solve_power() finds the group size needed for 80% power.

In [4]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()
power_exact = analysis.power(effect_size=d, nobs1=n, alpha=alpha)
n_needed = analysis.solve_power(effect_size=d, power=0.80, alpha=alpha)

print(f'exact power with {n} per group = {power_exact:.3f}')
print(f'n per group for 80% power = {n_needed:.2f}, so use {int(np.ceil(n_needed))}')

exact power with 36 per group = 0.388
n per group for 80% power = 99.08, so use 100


## Try this

- Change n to 100 and predict how the simulated and exact power change, then run it.
- Change alpha to 0.01 and predict whether the power goes up or down, then run it.

## The same idea in R, SPSS and Stata

**R**

```r
power.t.test(n = 36, delta = 6, sd = 15)                # power for 36 per group
power.t.test(power = 0.80, delta = 6, sd = 15)           # n per group for 80% power

# simulation (random, so close to but not exactly the Python value)
set.seed(42)
mean(replicate(10000, t.test(rnorm(36, 0, 15), rnorm(36, 0, 15))$p.value <= 0.05))
mean(replicate(10000, t.test(rnorm(36, 0, 15), rnorm(36, 6, 15))$p.value <= 0.05))
```

**SPSS** (Analyze > Power Analysis > Means > Independent-Samples T Test (enter power 0.80 to solve for the sample size))

```
DATA LIST FREE / n delta sd.
BEGIN DATA
36 6 15
END DATA.
COMPUTE df = 2 * n - 2.
COMPUTE nc = (delta / sd) * SQRT(n / 2).
COMPUTE tcrit = IDF.T(0.975, df).
COMPUTE power = 1 - NCDF.T(tcrit, df, nc) + NCDF.T(-tcrit, df, nc).
LIST power.
```

**Stata**

```
power twomeans 0 6, sd(15) n1(36) n2(36)      // power for 36 per group
power twomeans 0 6, sd(15) power(0.80)         // sample size for 80% power
```